# 第 7 章 · 多 agent（LoopAgent）

一个干活的 agent + 一个检查状态的自定义 agent，循环到条件满足或撞轮次上限。

停法和反射那章一样：**条件触发 `escalate=True`**，或 **`max_iterations` 硬停**。

⚠️ 书里那版永远停不下来：instruction 让模型"把 status 设成 completed"，**但 LlmAgent 写不了 state**。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

from google.adk.runners import InMemoryRunner
from google.genai import types


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

from typing import AsyncGenerator

from google.adk.agents import BaseAgent, LlmAgent, LoopAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event, EventActions
from google.adk.tools import ToolContext


class ConditionChecker(BaseAgent):
    """检查 state 里的 status 来决定要不要停。"""
    name: str = "ConditionChecker"
    description: str = "Checks if a process is complete and signals the loop to stop."

    async def _run_async_impl(self, context: InvocationContext) -> AsyncGenerator[Event, None]:
        status = context.session.state.get("status", "pending")
        if status == "completed":
            # ★ 本章新增 escalate：置为 True 就是"跳出循环"的信号
            yield Event(author=self.name, actions=EventActions(escalate=True))
        else:
            yield Event(author=self.name,
                        content=types.Content(role="model",
                                              parts=[types.Part(text=f"status={status}，继续")]))


def mark_done(tool_context: ToolContext) -> str:
    """任务完成时调用，把 session state 里的 status 标成 completed。"""
    # 【适配】书里让模型直接改 state —— LlmAgent 做不到，只能通过工具写
    tool_context.state["status"] = "completed"
    return "status set to completed"


process_step = LlmAgent(
    name="ProcessingStep",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="你是流程中的一步：复述一次当前目标，然后立刻调用 mark_done 表示完成。",
    tools=[mark_done],
)

poller = LoopAgent(name="StatusPoller", max_iterations=3, sub_agents=[process_step, ConditionChecker()])

finals, state = run_once(poller, "检查并完成这个任务")
print(f"共 {len(finals)} 个终态事件（= 轮数 × 子 agent 数）")
for author, text in finals:
    print(f"  [{author}] {text[:120]}")
print("\n结束时 state['status'] =", state.get("status"))

共 1 个终态事件（= 轮数 × 子 agent 数）
  [ProcessingStep] 任务已成功完成，目标已达成。

结束时 state['status'] = completed
